## Working with GEOREF in Vgrid DGGS

[![image](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://demo.vgrid.vn/lab/index.html?path=vgrid/13_georef.ipynb)
[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeoshub/vgrid/blob/main/docs/notebooks/13_georef.ipynb)
[![image](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/opengeoshub/vgrid/HEAD?filepath=docs/notebooks/13_georef.ipynb)
[![image](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/opengeoshub/vgrid/blob/main/docs/notebooks/13_georef.ipynb)
[![image](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://jupyterlite.gishub.vn/lab/index.html?path=notebooks/vgrid/13_georef.ipynb)

Full Vgrid DGGS documentation is available at [vgrid document](https://vgrid.gishub.vn).

To work with Vgrid DGGS directly in GeoPandas and Pandas, please use [vgridpandas](https://pypi.org/project/vgridpandas/). Full Vgridpandas DGGS documentation is available at [vgridpandas document](https://vgridpandas.gishub.vn).

To work with Vgrid DGGS in QGIS, install the [Vgrid Plugin](https://plugins.qgis.org/plugins/vgridtools/).

To visualize DGGS in Maplibre GL JS, try the [vgrid-maplibre](https://www.npmjs.com/package/vgrid-maplibre) library.

For an interactive demo, visit the [Vgrid Homepage](https://vgrid.vn).

### Install vgrid
Uncomment the following line to install [vgrid](https://pypi.org/project/vgrid/).

In [ ]:
# %pip install vgrid --upgrade

### latlon2georef

In [ ]:
from vgrid.conversion.latlon2dggs import latlon2georef

lat = 10.775276
lon = 106.706797
res = 4
georef_id = latlon2georef(lat, lon, res)
georef_id

### GEOREF to Polygon

In [ ]:
from vgrid.conversion.dggs2geo.georef2geo import georef2geo
from vgrid.dggs.georef import decode

georef_geo = georef2geo(georef_id)
georef_geo

### GEOREF to GeoJSON

In [ ]:
from vgrid.conversion.dggs2geo.georef2geo import georef2geojson

georef_geojson = georef2geojson(georef_id)
georef_geojson

### Vector to GEOREF

In [ ]:
from vgrid.conversion.vector2dggs.vector2georef import vector2georef

file_path = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/polygon2.geojson"
vector_to_georef = vector2georef(
    file_path,
    resolution=3,
    compact=False,
    topology=False,
    predicate="intersects",
    output_format="gpd",
)
# Visualize the output
# vector_to_georef
vector_to_georef.plot(edgecolor="white")

### GEOREF Binning

In [ ]:
from vgrid.binning.georefbin import georefbin

file_path = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/csv/housing.csv"
)
stats = "max"
numeric_col="median_house_value"

georef_bin = georefbin(
    file_path,
    resolution=1,
    stats=stats,
    numeric_col=numeric_col,
    # category_col="category",
    output_format="gpd",
)
georef_bin.plot(
    column= f"{numeric_col}_{stats}",  # numeric column to base the colors on
    cmap="Spectral_r",  # color scheme (matplotlib colormap)
    legend=True,
    linewidth=0.2,  # boundary width (optional)
)

### Raster to GEOREF

#### Download and open raster

In [ ]:
from vgrid.utils.io import download_file
import rasterio
from rasterio.plot import show

raster_url = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/raster/rgb.tif"
)
raster_file = download_file(raster_url)
src = rasterio.open(raster_file, "r")
print(src.meta)
show(src)

#### Convert raster to GEOREF

In [ ]:
# %pip install folium

In [ ]:
from vgrid.conversion.raster2dggs.raster2georef import raster2georef

raster_to_georef = raster2georef(raster_file, 
                                method = 'binning', # nearest, binning
                                stats = 'mean', output_format="gpd")

# Visualize the output
import folium

m = folium.Map(tiles="CartoDB positron", max_zoom=28)

georef_layer = folium.GeoJson(
    raster_to_georef,
    style_function=lambda x: {
        "fillColor": f"rgb({x['properties']['band_1']}, {x['properties']['band_2']}, {x['properties']['band_3']})",
        "fillOpacity": 1,
        "color": "black",
        "weight": 1,
    },
    popup=folium.GeoJsonPopup(
        fields=["georef", "band_1", "band_2", "band_3"],
        aliases=["GEOREF ID", "Band 1", "Band 2", "Band 3"],
        style="""
            background-color: white;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
    ),
).add_to(m)

m.fit_bounds(georef_layer.get_bounds())

# Display the map
m

### GEOREF Generator

In [ ]:
from vgrid.generator.georefgrid import georefgrid
georef_grid = georefgrid(resolution=0)
# georef_grid = georefgrid(resolution=1,bbox=[104.03833902,8.53410125,111.92366325,12.80076792])
georef_grid.plot(edgecolor="white")

### GEOREF Inspect

In [ ]:
from vgrid.stats.georefstats import georefinspect

resolution = 1
georef_inspect = georefinspect(resolution)
georef_inspect.head()

### GEOREF Normalized Area Histogram

In [ ]:
from vgrid.stats.georefstats import georef_norm_area_hist

georef_norm_area_hist(georef_inspect)

### Distribution of GEOREF Area Distortions

In [ ]:
from vgrid.stats.georefstats import georef_norm_area

georef_norm_area(georef_inspect)

### GEOREF IPQ Compactness Histogram

Isoperimetric Inequality (IPQ) Compactness (suggested by [Osserman, 1978](https://sites.math.washington.edu/~toro/Courses/20-21/MSF/osserman.pdf)):

$$C_{IPQ} = \frac{4 \pi A}{p^2}$$
The range of the IPQ compactness metric is [0,1]. 

A circle represents the maximum compactness with a value of 1. 

As shapes become more irregular or elongated, their compactness decreases toward 0.

In [ ]:
from vgrid.stats.georefstats import georef_compactness_ipq_hist

georef_compactness_ipq_hist(georef_inspect)

### Distribution of GEOREF IPQ Compactness

In [ ]:
from vgrid.stats.georefstats import georef_compactness_ipq

georef_compactness_ipq(georef_inspect)

### GEOREF Convex hull Compactness Histogram:

$$C_{CVH} = \frac{A}{A_{CVH}}$$


The range of the convex hull compactness metric is [0,1]. 

As shapes become more concave, their convex hull compactness decreases toward 0.

In [ ]:
from vgrid.stats.georefstats import georef_compactness_cvh_hist
georef_compactness_cvh_hist(georef_inspect)

### Distribution of GEOREF Convex hull Compactness

In [ ]:
from vgrid.stats.georefstats import georef_compactness_cvh
georef_compactness_cvh(georef_inspect)

### GEOREF Statistics

Characteristic Length Scale (CLS - suggested by Ralph Kahn): the diameter of a spherical cap of the same cell's area

In [ ]:
from vgrid.stats import georefstats

georefstats("m")